In [ ]:
from pathlib import Path

import fiona
from fiona.crs import from_epsg
import pystac
from shapely.geometry import shape, mapping
from functools import reduce
from pyproj import Transformer
from shapely.ops import transform

from tqdm.notebook import tqdm

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
from config import *

# -- Catalog ----------------------------------------------------------------------------------------------------------------------------------------------------------------------
CATALOG             = load_catalog()

# -- Output -----------------------------------------------------------------------------------------------------------------------------------------------------------------------
OUT_DIR             = Path("../out/bboxes")
MKDIR               = True
OUT_NAME            = "common-footprint"
OUT_EXT             = "geojson"

# -- Execution Flags -----------------------------------------------------------------------------------------------------------------------------------------------------------------------
OVERWRITE           = False

In [ ]:
wv2_col = CATALOG.get_child("wv2-imagery")
sd8_col = CATALOG.get_child("sd8-imagery")

items = list(sd8_col.get_items()) + list(wv2_col.get_items())

tqdm.write(f"{len(items)} item(s) found:")
for item in items:
    tqdm.write(f"ID: {item.id} | Date: {item.datetime} | Path: {item.assets['image'].href}")

In [ ]:
OUT_DIR.mkdir(parents = True, exist_ok=MKDIR)

In [ ]:
footprints = []

for item in items:
    if item.geometry:
        footprints.append(shape(item.geometry))

common_footprint = reduce(lambda a, b: a.intersection(b), footprints)

In [ ]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32756", always_xy=True)
common_footprint_32756 = transform(transformer.transform, common_footprint)

In [ ]:
schema = {"geometry": "Polygon", "properties": {"crs": "str", "scenes": "str"}}

out_path = OUT_DIR / f"{OUT_NAME}.{OUT_EXT}"

asset = pystac.Asset(
    href=str(out_path),
    media_type="application/geo+json",
    roles=["metadata"],
    title = "Common Footprint",
    description= "Common spatial footprint across WV2 and Planet scenes"
)

for item in items:
    item.add_asset(asset = asset, key = OUT_NAME)

if out_path.exists() and not OVERWRITE:
    tqdm.write(f"Skipping (exists): {out_path.name}")

else:
    with fiona.open(out_path, "w", driver="GeoJSON", crs=from_epsg(32756), schema=schema) as f:
        f.write({
            "geometry": mapping(common_footprint_32756),
            "properties": {
                "crs": "EPSG:32756",
                "scenes": ", ".join(item.id for item in items),
            }
        })

CATALOG.normalize_hrefs(str(STAC_DIR))
CATALOG.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)